In [3]:
import pandas as pd
import numpy as np

"""
Exercise 5: Pandas Data Analysis
Analyze a dataset of student performance.
"""

# Create sample dataset
np.random.seed(42)
n_students = 200

data = {
    'student_id': range(1000, 1000 + n_students),
    'major': np.random.choice(['CS', 'Math', 'Physics', 'Biology'], n_students),
    'year': np.random.choice([1, 2, 3, 4], n_students),
    'exam_score': np.random.normal(75, 10, n_students).clip(0, 100),
    'assignments_completed': np.random.randint(0, 11, n_students),
    'hours_studied': np.random.normal(15, 5, n_students).clip(1, 40)
}

df = pd.DataFrame(data)

# Introduce some NaN values
df.loc[np.random.choice(n_students, 10), 'exam_score'] = np.nan
df.loc[np.random.choice(n_students, 5), 'hours_studied'] = np.nan


# Task 1: Data Cleaning and Exploration (10 points)
# TODO: Display basic information about the dataset
# TODO: Identify and count missing values
# TODO: Fill missing exam_score with the mean score for the student's major
# TODO: Fill missing hours_studied with the median for the student's year

print("Task 1:")
# Display basic information
print("\nDataFrame Summary Info:")
df.info()

# Count missing values
print("\n Missing Values Count:")
print(df.isnull().sum())

for current_major in ['CS', 'Math', 'Physics', 'Biology']:
    # Step 1: Isolate rows matching the major and find the average score
    major_mask = df['major'] == current_major
    mean_score = df[major_mask]['exam_score'].mean()
# Step 2: Apply fillna only to that filtered slice of data
    df.loc[major_mask, 'exam_score'] = df[major_mask]['exam_score'].fillna(mean_score)

for current_year in [1, 2, 3, 4]:
  year_mask = df['year'] == current_year

#Finding the median
median_hours = df[year_mask]['hours_studied'].median()

df.loc[year_mask, 'hours_studied'] = df[year_mask]['hours_studied'].fillna(median_hours)

print("\nMissing values after cleaning:", df.isnull().sum().sum())


# Task 2: Analysis (10 points)
# TODO: Calculate and display the average exam_score by major
# TODO: Find the major with the highest average exam_score
# TODO: Calculate the correlation between hours_studied and exam_score
# TODO: Create a new column 'performance' with categories:
#       'Excellent' (>90), 'Good' (80-90), 'Average' (70-80), 'Needs Improvement' (<70)

print("Task 2:")
# Average exam_score by major
major_stats = df.groupby('major')['exam_score'].agg(['mean'])
print("Average Exam Score by Major: \n")
print(major_stats)

# Find major with the highest average score
sorted_majors = major_stats.sort_values('mean', ascending=False)
print("\nMajors sorted by highest score:")
print(sorted_majors)

# Correlation between hours_studied and exam_score
correlation_table = df[['hours_studied', 'exam_score']].corr()
print("\nCorrelation matrix between study hours and exam performance:")
print(correlation_table)

df['performance'] = 'Needs Improvement'

df.loc[(df['exam_score'] >= 70) & (df['exam_score'] <= 80), 'performance'] = 'Average'
df.loc[(df['exam_score'] > 80) & (df['exam_score'] <= 90), 'performance'] = 'Good'
df.loc[df['exam_score'] > 90, 'performance'] = 'Excellent'


print("Task 3:")
# calculate:
# Number of students
cohort_counts = df.groupby(['major', 'year'])['student_id'].count()

# Average exam score
cohort_avg_scores = df.groupby(['major', 'year'])['exam_score'].mean()

# Average hours studied
cohort_avg_hours = df.groupby(['major', 'year'])['hours_studied'].mean()

# Combining and Summary
cohort_dictionary = {
    'Number of Students': cohort_counts,
    'Average Exam Score': cohort_avg_scores,
    'Average Hours Studied': cohort_avg_hours
}
cohort_summary = pd.DataFrame(cohort_dictionary)
print("Grouped Summary")
print(cohort_summary.head())

# Identify top 5 students based on exam_score (including ties)
df_sorted = df.sort_values('exam_score', ascending=False)
top_five_rows = df_sorted.head(5)
top_students = top_five_rows[['student_id', 'major', 'exam_score']]
print("\nTop 5 Performing Students:")
print(top_students)

# Pivot table showing average exam_score by major (rows) and year (columns)
pivot_summary = df.pivot_table(
    index='major',      # Rows
    columns='year',     # Columns
    values='exam_score',
    aggfunc='mean'
)

print("\n Avg Exam Score by Major vs Year:")
print(pivot_summary)

Task 1:

DataFrame Summary Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             200 non-null    int64  
 1   major                  200 non-null    object 
 2   year                   200 non-null    int64  
 3   exam_score             190 non-null    float64
 4   assignments_completed  200 non-null    int64  
 5   hours_studied          195 non-null    float64
dtypes: float64(2), int64(3), object(1)
memory usage: 9.5+ KB

 Missing Values Count:
student_id                0
major                     0
year                      0
exam_score               10
assignments_completed     0
hours_studied             5
dtype: int64

Missing values after cleaning: 4
Task 2:
Average Exam Score by Major: 

              mean
major             
Biology  75.503150
CS       76.425184
Math     74.683689
Physics  75.